# Fine-tune Gemma 4 E4B-it on `dair-ai/emotion`

This notebook fine-tunes **`google/gemma-4-E4B-it`** on **`dair-ai/emotion`** 



## 1. Install dependencies

In [1]:
%%capture
!pip install -U transformers accelerate datasets trl peft bitsandbytes scikit-learn huggingface_hub

## 2. Login with `HF_TOKEN`

In [2]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Set HF_TOKEN in the RunPod environment before running this notebook.")

login(token=hf_token)
print("Logged in to Hugging Face.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.


## 3. Load the dataset

In [3]:
from datasets import load_dataset, DatasetDict

TRAIN_LIMIT = 4000     
VALIDATION_LIMIT = 400   
TEST_LIMIT = 400          
EVAL_LIMIT = 400         

raw_dataset = load_dataset("dair-ai/emotion")

def maybe_limit(split, limit):
    split = split.shuffle(seed=42)
    if limit is None:
        return split
    return split.select(range(min(limit, len(split))))

dataset = DatasetDict({
    "train": maybe_limit(raw_dataset["train"], TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
    "test": maybe_limit(raw_dataset["test"], TEST_LIMIT),
})

dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
})

In [4]:
label_names = dataset["train"].features["label"].names
label_names

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [5]:
dataset["train"][0]

{'text': 'while cycling in the country', 'label': 4}

## 4. Create the system prompt

In [6]:
SYSTEM_PROMPT = """You are an emotion classification assistant.
Read the user's text and answer with exactly one label.
Only choose from: sadness, joy, love, anger, fear, surprise.
Return only the label and nothing else."""

## 5. Convert the dataset into TRL prompt-completion chat format


In [7]:
def to_prompt_completion(example):
    text = example["text"]
    label = label_names[example["label"]]
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"Classify the emotion of this text:\n\n{text}",
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": label,
            }
        ],
    }

sft_dataset = dataset.map(to_prompt_completion, remove_columns=dataset["train"].column_names)


In [8]:
sft_dataset["train"][0]


{'prompt': [{'content': "You are an emotion classification assistant.\nRead the user's text and answer with exactly one label.\nOnly choose from: sadness, joy, love, anger, fear, surprise.\nReturn only the label and nothing else.",
   'role': 'system'},
  {'content': 'Classify the emotion of this text:\n\nwhile cycling in the country',
   'role': 'user'}],
 'completion': [{'content': 'fear', 'role': 'assistant'}]}

## 6. Load the processor and a base model for pre-fine-tuning evaluation

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B-it"
MODEL_DTYPE = torch.bfloat16
USE_4BIT = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

processor = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if processor.pad_token is None:
    processor.pad_token = processor.eos_token

bnb_config = None
model_kwargs = {
    "device_map": "auto",
}
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=MODEL_DTYPE,
    )
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["torch_dtype"] = MODEL_DTYPE

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
base_model.config.use_cache = False
base_model.config.pad_token_id = processor.pad_token_id
base_model.config.bos_token_id = processor.bos_token_id
base_model.config.eos_token_id = processor.eos_token_id
base_model.generation_config.pad_token_id = processor.pad_token_id
base_model.generation_config.bos_token_id = processor.bos_token_id
base_model.generation_config.eos_token_id = processor.eos_token_id

print(f"Base model loaded with 4-bit={USE_4BIT} and dtype={MODEL_DTYPE}.")


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Base model loaded with 4-bit=True and dtype=torch.bfloat16.


## 7. Inference helper for Gemma 4 chat formatting

In [10]:
import re

LABEL_PATTERN = re.compile(r"\b(sadness|joy|love|anger|fear|surprise)\b", re.IGNORECASE)

def extract_label(raw_text: str) -> str:
    raw_text = raw_text.strip().lower()
    match = LABEL_PATTERN.search(raw_text)
    if match:
        return match.group(1)

    first_token = raw_text.split()[0].strip(".,!?:;\"'()[]{}") if raw_text.split() else ""
    return first_token

def generate_label(model, processor, user_text, system_prompt, max_new_tokens=4):
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": f"Classify the emotion of this text:\n\n{user_text}",
        },
    ]

    device = next(model.parameters()).device
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.pad_token_id,
            eos_token_id=processor.eos_token_id,
        )

    raw_pred = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return extract_label(raw_pred)


In [11]:
def predict_emotion(user_text: str, model=None, proc=None) -> str:
    model = model or base_model
    proc = proc or processor
    return generate_label(model, proc, user_text, SYSTEM_PROMPT)

predict_emotion("I feel so happy and excited today!")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'joy'

## 8. Evaluation helpers

In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
from tqdm.auto import tqdm

VALID_LABELS = set(label_names)
ALL_EVAL_LABELS = label_names + ["INVALID"]

def evaluate_model(model, processor, split="test", limit=EVAL_LIMIT):
    y_true, y_pred, rows = [], [], []
    raw_source = dataset[split]
    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    model.eval()

    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        true_label = label_names[ex["label"]]
        raw_pred_label = generate_label(model, processor, ex["text"], SYSTEM_PROMPT)
        pred_label = raw_pred_label if raw_pred_label in VALID_LABELS else "INVALID"

        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({
            "text": ex["text"],
            "true_label": true_label,
            "pred_label": pred_label,
            "raw_pred_label": raw_pred_label,
            "correct": true_label == pred_label,
        })

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=label_names, average="macro", zero_division=0),
        "invalid_predictions": sum(1 for p in y_pred if p == "INVALID"),
        "evaluated_examples": len(y_true),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=label_names,
        output_dict=True,
        zero_division=0,
    )

    df = pd.DataFrame(rows)
    return metrics, report, df

def confusion_matrix_df(pred_df):
    return pd.DataFrame(
        confusion_matrix(pred_df["true_label"], pred_df["pred_label"], labels=ALL_EVAL_LABELS),
        index=ALL_EVAL_LABELS,
        columns=ALL_EVAL_LABELS,
    )


## 9. Pre-fine-tuning evaluation

In [13]:
pre_metrics, pre_report, pre_preds = evaluate_model(base_model, processor, "test")
pre_metrics


Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]

{'accuracy': 0.5825,
 'macro_f1': 0.42112912841373906,
 'invalid_predictions': 33,
 'evaluated_examples': 400}

In [14]:
pd.DataFrame(pre_report).transpose()

,precision,recall,f1-score,support
sadness,0.564706,0.842105,0.676056,114.0
joy,0.794326,0.691358,0.739274,162.0
love,0.285714,0.230769,0.255319,26.0
anger,0.600000,0.063830,0.115385,47.0
fear,0.666667,0.333333,0.444444,36.0
surprise,0.333333,0.266667,0.296296,15.0
micro avg,0.634877,0.582500,0.607562,400.0
macro avg,0.540791,0.404677,0.421129,400.0
weighted avg,0.644215,0.582500,0.573347,400.0


In [15]:
confusion_matrix_df(pre_preds)


,sadness,joy,love,anger,fear,surprise,INVALID
sadness,96,8,3,1,2,1,3
joy,25,112,7,1,2,4,11
love,8,11,6,0,0,0,1
anger,22,3,2,3,2,0,15
fear,14,4,1,0,12,3,2
surprise,5,3,2,0,0,4,1
INVALID,0,0,0,0,0,0,0


## 10. Configure the LoRA adapters

In [16]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)


## 11. Define the training configuration

In [17]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./gemma4-emotion-lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    tf32=True,
    max_length=256,
    packing=False,
    completion_only_loss=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    report_to="none",
)


## 12. Train the model

In [18]:
from peft import PeftModel

if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=processor,
)

trainable_params = 0
for param in trainer.model.parameters():
    if param.requires_grad:
        trainable_params += param.numel()

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters were attached. Check target_modules before training.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
train_result = trainer.train()
trainer.model.eval()
trainer.model.config.use_cache = True
train_result


Trainable LoRA parameters: 52,318,208


Step,Training Loss,Validation Loss
50,9.834910,0.499835
100,0.914290,0.409208
150,0.644271,0.286898
200,0.455626,0.240309
250,0.381435,0.190164


TrainOutput(global_step=250, training_loss=2.4461064376831056, metrics={'train_runtime': 558.1942, 'train_samples_per_second': 7.166, 'train_steps_per_second': 0.448, 'total_flos': 1.2144898594450944e+16, 'train_loss': 2.4461064376831056})

## 13. Save the adapter and processor

In [19]:
trainer.model.save_pretrained("./gemma4-emotion-lora")
processor.save_pretrained("./gemma4-emotion-lora")
print("Saved adapter and processor to ./gemma4-emotion-lora")

Saved adapter and processor to ./gemma4-emotion-lora


In [20]:
repo_id = "kingabzpro/gemma4-emotion-lora"

# Push adapter + processor to the Hub
trainer.model.push_to_hub(
    repo_id,
    private=False,
)

processor.push_to_hub(
    repo_id,
    private=False,
)

print(f"Pushed to https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/kingabzpro/gemma4-emotion-lora


## 14. Run post-fine-tuning evaluation without reloading the model


In [21]:
# Reuse the in-memory fine-tuned model to avoid a second base-model load.
# On smaller GPUs, reloading after training often causes fragmentation or OOMs.
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True
post_metrics, post_report, post_preds = evaluate_model(ft_model, processor, "test")
post_metrics


Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]

{'accuracy': 0.7725,
 'macro_f1': 0.697702361480462,
 'invalid_predictions': 20,
 'evaluated_examples': 400}

In [22]:
pd.DataFrame(post_report).transpose()

,precision,recall,f1-score,support
sadness,0.871795,0.894737,0.883117,114.0
joy,0.909091,0.802469,0.852459,162.0
love,0.513514,0.730769,0.603175,26.0
anger,0.718750,0.489362,0.582278,47.0
fear,0.717949,0.777778,0.746667,36.0
surprise,0.583333,0.466667,0.518519,15.0
micro avg,0.813158,0.772500,0.792308,400.0
macro avg,0.719072,0.693630,0.697702,400.0
weighted avg,0.820965,0.772500,0.791203,400.0


In [23]:
confusion_matrix_df(post_preds)


,sadness,joy,love,anger,fear,surprise,INVALID
sadness,102,2,1,2,3,1,3
joy,5,130,16,4,3,3,1
love,1,6,19,0,0,0,0
anger,6,1,1,23,3,0,13
fear,2,0,0,2,28,1,3
surprise,1,4,0,1,2,7,0
INVALID,0,0,0,0,0,0,0


## 15. Compare before vs after

In [24]:
comparison_df = pd.DataFrame([
    {"stage": "pre_finetuning", **pre_metrics},
    {"stage": "post_finetuning", **post_metrics},
])
comparison_df


,stage,accuracy,macro_f1,invalid_predictions,evaluated_examples
0,pre_finetuning,0.5825,0.421129,33,400
1,post_finetuning,0.7725,0.697702,20,400


In [25]:
merged_examples = pre_preds.copy()
merged_examples = merged_examples.rename(columns={"pred_label": "pre_pred", "correct": "pre_correct"})
merged_examples["post_pred"] = post_preds["pred_label"]
merged_examples["post_correct"] = post_preds["correct"]

changed_predictions = merged_examples[merged_examples["pre_pred"] != merged_examples["post_pred"]]
changed_predictions.head(20)


,text,true_label,pre_pred,raw_pred_label,pre_correct,post_pred,post_correct
2,i feel is that the most likeable characters ar...,joy,sadness,sadness,False,love,False
5,im feeling and if ive liked being pregnant,love,joy,joy,False,love,True
7,i used to be able to hang around talk with the...,anger,sadness,sadness,False,INVALID,False
8,i don t have the feeling of divine vibrations,joy,sadness,sadness,False,joy,True
9,i vented my feelings towards the pathetic excu...,sadness,anger,anger,False,sadness,True
10,i feel privileged to be a part of something so...,joy,joy,joy,True,love,False
12,i feel like some of you have pains and you can...,joy,sadness,sadness,False,love,False
13,i cant write a review for a book i adore unles...,love,joy,joy,False,love,True
14,i mentioned in my last blog that i have starte...,fear,sadness,sadness,False,INVALID,False
16,i was wondering if you will focus on the probl...,sadness,INVALID,angerequ,False,sadness,True


## 16. Use the in-memory fine-tuned model for prediction


In [26]:
def predict_emotion_ft(user_text: str) -> str:
    return generate_label(ft_model, processor, user_text, SYSTEM_PROMPT)

predict_emotion_ft("I feel completely heartbroken and alone.")


'sadness'

In [27]:
predict_emotion_ft("This is the best day of my life!")

'joy'

## 17. Optional: save comparison results

In [28]:
comparison_df.to_csv("gemma4_emotion_before_after_metrics.csv", index=False)
merged_examples.to_csv("gemma4_emotion_prediction_examples.csv", index=False)
print("Saved CSV outputs.")

Saved CSV outputs.
